In [34]:
import os, cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [35]:
data_dir = "./"

X, y = [], []

# Collect cats
cat_files = [f for f in os.listdir(data_dir) if f.startswith("cat") and f.endswith(".jpg")]
for img_name in cat_files[:2000]:
    img_path = os.path.join(data_dir, img_name)
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    img = cv2.resize(img, (64, 64))
    X.append(img.flatten())
    y.append(0)

# Collect dogs
dog_files = [f for f in os.listdir(data_dir) if f.startswith("dog") and f.endswith(".jpg")]
for img_name in dog_files[:2000]:
    img_path = os.path.join(data_dir, img_name)
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    img = cv2.resize(img, (64, 64))
    X.append(img.flatten())
    y.append(1)

X = np.array(X)
y = np.array(y)
print("Dataset shape:", X.shape, "Labels:", len(y))


Dataset shape: (2573, 12288) Labels: 2573


In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [37]:
# PCA to reduce dimensionality
pca = PCA(n_components=500)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled = scaler.transform(X_test_pca)


In [38]:
clf = SVC(kernel='rbf', gamma='scale', C=10)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.7475728155339806
              precision    recall  f1-score   support

           0       0.77      0.95      0.85       395
           1       0.33      0.08      0.13       120

    accuracy                           0.75       515
   macro avg       0.55      0.52      0.49       515
weighted avg       0.67      0.75      0.68       515

Confusion Matrix:
 [[375  20]
 [110  10]]


In [39]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(clf, X_train_scaled, y_train, cv=5)
print("Cross-validation scores:", scores)
print("Mean CV accuracy:", scores.mean())


Cross-validation scores: [0.74029126 0.76699029 0.76456311 0.77128954 0.7810219 ]
Mean CV accuracy: 0.7648312191434579


In [40]:
import joblib
joblib.dump(clf, "svm_cats_vs_dogs.pkl")


['svm_cats_vs_dogs.pkl']